# Running Models — Audio · Part 4 — Companion Notebook

This goes with **"Narrator in a Box: Read Any Article Aloud"**. Run cells top to bottom (Shift+Enter). A long article on CPU is slow — for speed, attach a GPU: Runtime → Change runtime type → GPU.

## Install

`transformers` for the model, `datasets` for the voice embedding, `soundfile` to save a `.wav`, `gradio` for the UI, and `sentencepiece` because SpeechT5's processor needs it.

In [ ]:
!pip install -q transformers datasets soundfile gradio sentencepiece

## Load SpeechT5 once

Processor (tokenizer), model (encoder + spectrogram), vocoder (HiFi-GAN), plus the 256-D speaker fingerprint. Load these **once** and reuse them for every sentence.

In [ ]:
import torch, numpy as np
from transformers import SpeechT5Processor, SpeechT5ForTextToSpeech, SpeechT5HifiGan
from datasets import load_dataset

processor = SpeechT5Processor.from_pretrained("microsoft/speecht5_tts")
model = SpeechT5ForTextToSpeech.from_pretrained("microsoft/speecht5_tts")
vocoder = SpeechT5HifiGan.from_pretrained("microsoft/speecht5_hifigan")

embeddings = load_dataset("Matthijs/cmu-arctic-xvectors", split="validation")
speaker = torch.tensor(embeddings[7306]["xvector"]).unsqueeze(0)

## The narrator: split, synthesize, stitch

Split into sentences (the lookbehind keeps the punctuation), synthesize each one, append 0.3 s of silence (4,800 samples at 16 kHz), and concatenate into one waveform. Returns `(sample_rate, samples)`.

In [ ]:
import re

def narrate(article: str):
    sentences = re.split(r"(?<=[.!?])\s+", article.strip())
    clips = []
    for s in sentences:
        if not s:
            continue
        inputs = processor(text=s, return_tensors="pt")
        clip = model.generate_speech(inputs["input_ids"], speaker, vocoder=vocoder)
        clips.append(clip.numpy())
        clips.append(np.zeros(int(0.3 * 16000)))   # 0.3 s pause between sentences
    return 16000, np.concatenate(clips)

## Try it

Three sentences in, one clip out, with a small breath between each.

In [ ]:
sr, audio = narrate(
    "Audio is just a tensor. A model turns text into a spectrogram. "
    "A vocoder turns that spectrogram into sound."
)
from IPython.display import Audio
Audio(audio, rate=sr)

## Wrap it in a Gradio UI

`narrate` already returns `(sample_rate, samples)` — exactly what `gr.Audio` wants — so it drops straight in. `.launch()` prints a public share link in Colab.

In [ ]:
import gradio as gr

gr.Interface(
    fn=narrate,
    inputs=gr.Textbox(lines=8, label="Paste an article"),
    outputs=gr.Audio(label="Narration"),
    title="Narrator in a Box",
).launch()

## Next

**Part 5 — How Audio Becomes Tokens: Neural Codecs and Encodec** — the trick that lets a language-model-shaped network generate sound, one token at a time.